In [4]:
#  get the code (push local changes to GitHub first)
# delete any previous clone so every run starts from a fresh copy
%cd /content
!rm -rf /content/nepali-summarizer
!git clone https://github.com/tekraj/nepali-summarizer.git /content/nepali-summarizer
%cd /content/nepali-summarizer

/content
Cloning into '/content/nepali-summarizer'...
remote: Enumerating objects: 266040, done.
remote: Counting objects: 100% (113589/113589), done.
remote: Compressing objects: 100% (113416/113416), done.
remote: Total 266040 (delta 187), reused 113546 (delta 172), pack-reused 152451 (from 2)
Receiving objects: 100% (266040/266040), 177.02 MiB | 16.77 MiB/s, done.
Resolving deltas: 100% (2259/2259), done.
Updating files: 100% (265187/265187), done.
/content/nepali-summarizer


In [5]:
# install dependencies from pyproject.toml (keeps Colab's CUDA torch)
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nepali-summarizer (pyproject.toml) ... done


In [ ]:
# train the model on Colab's GPU, keeping checkpoints on Drive so a disconnect loses at most
# --save-every batches. Re-run this cell after a disconnect (after cells 1-2): it resumes
# from the newest checkpoint on Drive, with Adam state, epoch, batch and running loss.
# To continue from a local checkpoint (e.g. summarizer_epoch_8.npz), upload it to
# MyDrive/nepali-summarizer/checkpoints/ first.
import os
import re
import shutil
from pathlib import Path

import numpy as np
from google.colab import drive

# The model runs on CuPy, which needs an NVIDIA GPU; a CPU runtime has no nvidia-smi.
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU in this runtime: Runtime -> Change runtime type -> T4 GPU, then re-run cells 1-2")

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/nepali-summarizer")
os.chdir(PROJECT_ROOT)

DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/nepali-summarizer/checkpoints")
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# config.yaml saves to <project>/checkpoints, so point that folder at Drive.
local_checkpoints = PROJECT_ROOT / "checkpoints"
if local_checkpoints.is_symlink():
    local_checkpoints.unlink()
elif local_checkpoints.exists():
    raise RuntimeError(f"{local_checkpoints} is a real folder; move its files to {DRIVE_CHECKPOINTS} first")
local_checkpoints.symlink_to(DRIVE_CHECKPOINTS, target_is_directory=True)

BATCH_SIZE = None  # None = config.yaml (16, fp16). A mid-epoch resume converts a checkpoint saved with another batch size.
SAVE_EVERY = 500  # batches between mid-epoch checkpoints (500 x 16 = the same 8000 samples as 2000 x 4)


def checkpoint_order(path: Path):
    """(epoch, step) for summarizer_epoch_E.npz / summarizer_epoch_E_step_S.npz, else None.
    An end-of-epoch file counts as later than every _step_ file of the same epoch."""
    match = re.fullmatch(r"summarizer_epoch_(\d+)(?:_step_(\d+))?", path.stem)
    if match is None:
        return None
    return int(match[1]), float("inf") if match[2] is None else int(match[2])


candidates = [p for p in DRIVE_CHECKPOINTS.glob("summarizer_epoch_*.npz") if checkpoint_order(p)]
args = ["--save-every", str(SAVE_EVERY)]
if BATCH_SIZE is not None:
    args += ["--batch-size", str(BATCH_SIZE)]

if candidates:
    latest = max(candidates, key=checkpoint_order)
    args += ["--resume", str(latest)]
    with np.load(latest) as saved:
        has_state = "state.epoch" in saved.files
    if not has_state:
        # Weights-only file from before training state was saved: give the position by hand.
        epoch, step = checkpoint_order(latest)
        args += ["--start-epoch", str(epoch + 1)] if step == float("inf") else [
            "--start-epoch", str(epoch), "--skip-steps", str(step)]
    print(f"Resuming from {latest.name}" + ("" if has_state else " (weights only, Adam restarts)"))
else:
    print("No checkpoint on Drive: training from scratch")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
train_args = " ".join(args)
print("python -m scripts.train_model", train_args)
!python -m scripts.train_model {train_args}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No checkpoint on Drive: training from scratch
Tesla T4, 15360 MiB
python -m scripts.train_model --save-every 500
Encoder: V=30000 D=512 H=8 N=6 D_ff=2048 T<=768+256 B=4 | 49.7M parameters
train: 458batch [02:33,  2.94batch/s, T=974, avg=4.7195, grad=1.33, loss=4.1340]  